In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression


# =========================================================
# 1. 데이터 불러오기
# =========================================================

train = pd.read_csv(
    "/kaggle/input/competitions/titanic/train.csv"
)

test = pd.read_csv(
    "/kaggle/input/competitions/titanic/test.csv"
)

y = train["Survived"].astype(int)


# =========================================================
# 2. 기본 변수
# =========================================================

numeric_features = [
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked"
]

features = (
    numeric_features
    + categorical_features
)


# =========================================================
# 3. 기본 전처리기
# Random Forest와 ExtraTrees에서 사용
# =========================================================

tree_preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        ),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]),
        categorical_features
    )
])


# =========================================================
# 4. HistGradientBoosting 전처리기
# 밀집 행렬 필요
# =========================================================

hist_preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        ),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]),
        categorical_features
    )
])


# =========================================================
# 5. Logistic Regression 전처리기
# =========================================================

logistic_preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]),
        categorical_features
    )
])


# =========================================================
# 6. 네 가지 모델
# =========================================================

rf_model = Pipeline([
    (
        "preprocessor",
        tree_preprocessor
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=500,
            max_depth=6,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1
        )
    )
])

extra_model = Pipeline([
    (
        "preprocessor",
        tree_preprocessor
    ),
    (
        "model",
        ExtraTreesClassifier(
            n_estimators=800,
            max_depth=8,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1
        )
    )
])

hist_model = Pipeline([
    (
        "preprocessor",
        hist_preprocessor
    ),
    (
        "model",
        HistGradientBoostingClassifier(
            max_iter=250,
            learning_rate=0.05,
            max_leaf_nodes=15,
            min_samples_leaf=20,
            l2_regularization=1.0,
            random_state=42
        )
    )
])

logistic_model = Pipeline([
    (
        "preprocessor",
        logistic_preprocessor
    ),
    (
        "model",
        LogisticRegression(
            C=1.0,
            max_iter=2000,
            solver="liblinear",
            random_state=42
        )
    )
])


# =========================================================
# 7. 모델 학습
# =========================================================

models = {
    "RandomForest": rf_model,
    "ExtraTrees": extra_model,
    "HistGradientBoosting": hist_model,
    "LogisticRegression": logistic_model
}

model_probabilities = {}

for model_name, model in models.items():
    model.fit(
        train[features],
        y
    )

    model_probabilities[
        model_name
    ] = model.predict_proba(
        test[features]
    )[:, 1]


# =========================================================
# 8. 가족 키 생성
# =========================================================

def make_family_key(data):
    surname = (
        data["Name"]
        .str.split(",")
        .str[0]
        .str.strip()
    )

    family_size = (
        data["SibSp"]
        + data["Parch"]
        + 1
    )

    family_key = (
        surname
        + "_"
        + family_size.astype(str)
    )

    return family_key.where(
        family_size > 1,
        np.nan
    )


train_family_key = make_family_key(train)
test_family_key = make_family_key(test)


# =========================================================
# 9. 여성·16세 미만 가족 통계
# =========================================================

train_protected_group = (
    (train["Sex"] == "female")
    | (train["Age"] < 16)
)

test_protected_group = (
    (test["Sex"] == "female")
    | (test["Age"] < 16)
)

family_data = pd.DataFrame({
    "FamilyKey": train_family_key,
    "Survived": y,
    "ProtectedGroup": train_protected_group
})

family_data = family_data[
    family_data["ProtectedGroup"]
].dropna(
    subset=["FamilyKey"]
)

family_statistics = (
    family_data
    .groupby("FamilyKey")["Survived"]
    .agg(["sum", "count"])
)


# =========================================================
# 10. 가족 생존율 Smoothing
# =========================================================

global_survival_rate = y.mean()

family_survival_rate = (
    family_statistics["sum"]
    + 2 * global_survival_rate
) / (
    family_statistics["count"]
    + 2
)

test_family_probability = (
    test_family_key
    .map(family_survival_rate)
)

use_family_information = (
    test_protected_group
    & test_family_probability.notna()
)


# =========================================================
# 11. Master 규칙 준비
# =========================================================

all_survived_family_keys = set(
    family_statistics.index[
        family_statistics["sum"]
        == family_statistics["count"]
    ]
)

test_title = (
    test["Name"]
    .str.extract(
        r",\s*([^.]*)\.",
        expand=False
    )
    .str.strip()
)


# =========================================================
# 12. 각 모델에 가족 정보와 Master 규칙 적용
# =========================================================

def apply_family_rules(
    base_probability
):
    final_probability = (
        base_probability.copy()
    )

    final_probability[
        use_family_information
    ] = (
        0.5
        * base_probability[
            use_family_information
        ]
        + 0.5
        * test_family_probability[
            use_family_information
        ].to_numpy()
    )

    prediction = (
        final_probability >= 0.5
    ).astype(int)

    rescue_master = (
        (test["Sex"] == "male")
        & (test_title == "Master")
        & test_family_key.isin(
            all_survived_family_keys
        )
        & (
            (test["Age"] < 13)
            | test["Age"].isna()
        )
        & (prediction == 0)
    )

    prediction[
        rescue_master
    ] = 1

    return prediction


rf_prediction = apply_family_rules(
    model_probabilities[
        "RandomForest"
    ]
)

extra_prediction = apply_family_rules(
    model_probabilities[
        "ExtraTrees"
    ]
)

hist_prediction = apply_family_rules(
    model_probabilities[
        "HistGradientBoosting"
    ]
)

logistic_prediction = apply_family_rules(
    model_probabilities[
        "LogisticRegression"
    ]
)


# =========================================================
# 13. 4개 모델 다수결
# =========================================================

model_predictions = np.column_stack([
    rf_prediction,
    extra_prediction,
    hist_prediction,
    logistic_prediction
])

survival_votes = (
    model_predictions.sum(axis=1)
)

# 생존표가 3개 이상이면 생존
ensemble_prediction = (
    survival_votes >= 3
).astype(int)

# 2대2 동률이면 ExtraTrees 선택
tie = (
    survival_votes == 2
)

ensemble_prediction[
    tie
] = extra_prediction[
    tie
]


# =========================================================
# 14. 제출 파일 생성
# =========================================================

submission = pd.DataFrame({
    "PassengerId":
        test["PassengerId"].astype(int),

    "Survived":
        ensemble_prediction.astype(int)
})

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)


# =========================================================
# 15. 결과 확인
# =========================================================

print(
    "제출 파일 크기:",
    submission.shape
)

print(
    "최종 생존 예측:",
    int(ensemble_prediction.sum())
)

print(
    "4개 모델 전원 동의:",
    int(
        (
            (survival_votes == 0)
            | (survival_votes == 4)
        ).sum()
    )
)

print(
    "3대1 의견:",
    int(
        (
            (survival_votes == 1)
            | (survival_votes == 3)
        ).sum()
    )
)

print(
    "2대2 동률:",
    int(tie.sum())
)

print(
    "RF와 최종 예측이 다른 승객:",
    int(
        (
            rf_prediction
            != ensemble_prediction
        ).sum()
    )
)

print(
    "저장 위치:",
    "/kaggle/working/submission.csv"
)

display(
    submission.head()
)